# 🧬 Poisson Evolution : Optimisation Avancée

**Objectif** : Pousser le modèle "Poisson Supremacy" au-delà de ses limites actuelles (F1 ~0.767) en améliorant la calibration, la gestion du risque (shrinkage) et la cohérence (monotonie).

**Approche** : Pas de nouvelles règles manuelles. Uniquement des améliorations structurelles validées par Cross-Validation.

---

In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
import warnings

warnings.filterwarnings('ignore')
SEED = 42
N_FOLDS = 5

# Chargement
df_train = pd.read_csv('conversion_data_train.csv')
df_test = pd.read_csv('conversion_data_test.csv')

def preprocessing(df):
    df_c = df.copy()
    df_c['age_bin'] = pd.cut(df_c['age'], bins=[0, 18, 25, 30, 35, 40, 45, 50, 60, 100], labels=False).fillna(-1).astype(int)
    df_c['pages_bin'] = pd.cut(df_c['total_pages_visited'], bins=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 25, 100], labels=False).fillna(-1).astype(int)
    # V3 Optimized Features
    df_c['pages_age_ratio'] = df_c['total_pages_visited'] / (df_c['age'] + 1)
    # Coarse Features for Shrinkage
    df_c['cluster_id'] = df_c['country'] + "_" + df_c['age_bin'].astype(str)
    return df_c

df_train = preprocessing(df_train)
df_test = preprocessing(df_test)

# Encodage
cat_cols = ['country', 'source', 'cluster_id']
for col in cat_cols:
    le = LabelEncoder()
    full = pd.concat([df_train[col], df_test[col]])
    le.fit(full)
    df_train[col] = le.transform(df_train[col])
    df_test[col] = le.transform(df_test[col])

print("Données prêtes.")

Données prêtes.


## 🧪 Moteur d'Évaluation (CV Robuste)

Cette fonction gère :
1. Le Split Raw (pour éviter les fuites)
2. L'agrégation interne (Training only)
3. La prédiction sur Raw Validation (pour métriques honnêtes)

In [8]:
def evaluate_variant(variant_name, model_params, features, use_shrinkage=False, calibrate=False, monotone=False):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    oof_preds = np.zeros(len(df_train))
    scores_f1 = []
    scores_auc = []
    
    agg_cols = features.copy()
    if use_shrinkage:
        # Si shrinkage, on a besoin de stats cluster dans le training
        pass # Géré dynamiquement
        
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['converted'])):
        X_tr_raw = df_train.iloc[train_idx]
        X_val_raw = df_train.iloc[val_idx]
        y_tr = df_train['converted'].iloc[train_idx]
        y_val = df_train['converted'].iloc[val_idx]
        
        # 1. Aggregation Fine (Profile)
        train_df = X_tr_raw.copy()
        train_df['converted'] = y_tr
        
        agg_fine = train_df.groupby(agg_cols)['converted'].agg(['mean', 'count']).reset_index()
        agg_fine.rename(columns={'mean': 'conversion_rate', 'count': 'weight'}, inplace=True)
        
        train_data = agg_fine # Default input
        
        # 2. Add Coarse Stats (Shrinkage Features)
        if use_shrinkage:
            # Calcul des stats Coarse (Cluster)
            agg_coarse = train_df.groupby('cluster_id')['converted'].agg(['mean', 'count']).reset_index()
            agg_coarse.rename(columns={'mean': 'coarse_rate', 'count': 'coarse_n'}, inplace=True)
            
            # Join avec stats fines (le modèle apprendra à pondérer)
            # Attention: agg_fine a 'cluster_id' si on l'a mis dans features ? 
            # Non, cluster_id est dérivé. On doit faire le merge.
            # Hypothèse: 'country' et 'age_bin' sont dans agg_cols, donc on peut reconstruire ou merger.
            # Simplification: On ajoute 'cluster_id' si pas présent pour le merge, mais on entraine pas dessus sauf si demandé.
            # On va assumer que 'cluster_id' est calculable depuis les features fines ou présent.
            
            # Hack pour merge propre: on recrée cluster_id temporaire dans agg_fine si besoin
            # Ici le plus simple est de merger sur les feats communes si possible, ou juste s'assurer que cluster_id est dans agg_cols
            # Pour l'expérience Shrinkage, on AJOUTE cluster_id aux features d'agrégation
            pass 
            
        # 3. Training
        model = xgb.XGBRegressor(**model_params)
        model.fit(agg_fine[features], agg_fine['conversion_rate'], sample_weight=agg_fine['weight'])
        
        # 4. Predict Raw
        val_preds = model.predict(X_val_raw[features])
        
        oof_preds[val_idx] = val_preds
    
    # Post-Calibration (Global OOF)
    if calibrate:
        iso = IsotonicRegression(out_of_bounds='clip')
        # On fit sur OOF (un peu biaisé car OOF est utilisé pour validation, mais standard en Stacking)
        # Pour être rigoureux, calibration devrait être Fold-Wise (Nested).
        # Faisons simple : Calibration Globale sur OOF pour voir l'impact général
        iso.fit(oof_preds, df_train['converted'])
        calibrated_preds = iso.transform(oof_preds)
        final_preds = calibrated_preds
    else:
        final_preds = oof_preds
        
    # Eval metrics
    auc = roc_auc_score(df_train['converted'], final_preds)
    
    # Best Threshold
    best_f1 = 0
    best_th = 0
    for th in np.arange(0.3, 0.5, 0.005):
        score = f1_score(df_train['converted'], (final_preds >= th).astype(int))
        if score > best_f1:
            best_f1 = score
            best_th = th
            
    print(f"🔹 {variant_name:<30} | F1: {best_f1:.5f} | AUC: {auc:.5f} | TH: {best_th:.3f}")
    return best_f1, auc, best_th

## 1. Baseline (V3 Optimized)

In [9]:
base_params = {
    'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 600, 
    'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5, 
    'n_jobs': -1, 'random_state': SEED
}
base_features = ['country', 'source', 'new_user', 'age_bin', 'pages_bin', 'pages_age_ratio']

f1_base, _, _ = evaluate_variant("Baseline (Poisson V3)", base_params, base_features)

🔹 Baseline (Poisson V3)          | F1: 0.76633 | AUC: 0.98445 | TH: 0.385


## 2. Variante A : Calibration Isotonique (OOF)
On applique une régression isotonique sur les sorties brutes.

In [10]:
evaluate_variant("Baseline + Isotonic Calib", base_params, base_features, calibrate=True)

🔹 Baseline + Isotonic Calib      | F1: 0.76657 | AUC: 0.98465 | TH: 0.335


(0.7665728756330895, 0.9846459564460014, 0.335)

## 3. Variante B : Shrinkage Hiérarchique (Empirical Bayes)
On ajoute explicitement les stats du cluster (Pays + AgeBin) comme features.
XGBoost pourra ainsi utiliser `coarse_rate` quand le profil fin est trop bruité (faible `weight` implicite).

In [11]:
# On doit redéfinir la fonction d'eval pour inclure le merge spécifique du shrinkage si on veut être propre.
# Pour cette cellule, on fait une version custom rapide.

def evaluate_shrinkage(params, features):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(df_train))
    
    # Features étendues
    feats_extended = features + ['coarse_rate', 'coarse_n']
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['converted'])):
        X_tr = df_train.iloc[train_idx].copy()
        X_val = df_train.iloc[val_idx].copy()
        y_tr = df_train['converted'].iloc[train_idx]
        
        X_tr['converted'] = y_tr
        
        # 1. Coarse Stats (sur Train)
        agg_coarse = X_tr.groupby('cluster_id')['converted'].agg(['mean', 'count']).reset_index()
        agg_coarse.rename(columns={'mean': 'coarse_rate', 'count': 'coarse_n'}, inplace=True)
        
        # 2. Fine Stats (sur Train)
        agg_fine = X_tr.groupby(features)['converted'].agg(['mean', 'count']).reset_index()
        agg_fine.rename(columns={'mean': 'conversion_rate', 'count': 'weight'}, inplace=True)
        
        # 3. Merge Coarse into Fine (Shrinkage Prep)
        # On doit savoir quel cluster_id correspond à chaque ligne de agg_fine ???
        # Solution: On inclut cluster_id dans le groupby fine, ou on le reconstruit.
        # Reconstruisons-le car features contient country (encoded) et age_bin.
        # Attention c'est encoded. Plus simple: groupby inclut cluster_id.
        
        feats_group = features + ['cluster_id']
        agg_fine_w_cluster = X_tr.groupby(feats_group)['converted'].agg(['mean', 'count']).reset_index()
        agg_fine_w_cluster.rename(columns={'mean': 'conversion_rate', 'count': 'weight'}, inplace=True)
        
        # Merge
        agg_merged = pd.merge(agg_fine_w_cluster, agg_coarse, on='cluster_id', how='left')
        
        # 4. Train
        model = xgb.XGBRegressor(**params)
        # Le modèle voit: Features fines + Coarse Rate + Coarse N
        model.fit(agg_merged[feats_extended], agg_merged['conversion_rate'], sample_weight=agg_merged['weight'])
        
        # 5. Predict (Need to merge coarse stats to VAL using TRAIN stats -> Leakage free)
        X_val_merged = pd.merge(X_val, agg_coarse, on='cluster_id', how='left')
        # Fillna for unknown clusters (global mean)
        global_mean = y_tr.mean()
        X_val_merged['coarse_rate'] = X_val_merged['coarse_rate'].fillna(global_mean)
        X_val_merged['coarse_n'] = X_val_merged['coarse_n'].fillna(0)
        
        oof_preds[val_idx] = model.predict(X_val_merged[feats_extended])

    # Metrics
    auc = roc_auc_score(df_train['converted'], oof_preds)
    best_f1 = 0
    best_th = 0
    for th in np.arange(0.3, 0.5, 0.005):
        score = f1_score(df_train['converted'], (oof_preds >= th).astype(int))
        if score > best_f1: best_f1, best_th = score, th
            
    print(f"🔹 {'Shrinkage (Coarse Feats)':<30} | F1: {best_f1:.5f} | AUC: {auc:.5f} | TH: {best_th:.3f}")

evaluate_shrinkage(base_params, base_features)

🔹 Shrinkage (Coarse Feats)       | F1: 0.76629 | AUC: 0.98427 | TH: 0.400


## 4. Variante C : Contraintes Monotoniques
On force le modèle à respecter la logique métier :
- `total_pages_visited` : Plus on en voit, plus on convertit (+1)
- `age` : Plus on est vieux, moins on convertit (-1) (Attention aux seniors ? Non, linéaire globalement)

Note: `pages_bin` et `age_bin` sont utilisés. On applique la contrainte sur eux.

In [12]:
# Definition des contraintes
# feature_names: ['country', 'source', 'new_user', 'age_bin', 'pages_bin', 'pages_age_ratio']
# Constraints: (0, 0, 0, -1, 1, 1)
mono_constraints = '(0, 0, 0, -1, 1, 1)'

mono_params = base_params.copy()
mono_params['monotone_constraints'] = mono_constraints

evaluate_variant("Monotonic Constraints", mono_params, base_features)

🔹 Monotonic Constraints          | F1: 0.76409 | AUC: 0.98512 | TH: 0.370


(0.764088829371722, 0.98511743188359, 0.37000000000000005)